## Youtube chatbot using LangChain

## Install libraries

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

d:\langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1a - Indexing (Document Ingestion)

In [4]:
video_id = "Gfr50f6ZBvo"  # only the ID, not full URL

try:
    transcript_data = ytt_api.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript_list = [{"text": s.text, "start": s.start} for s in transcript_data]
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except Exception as e:
    print(f"No captions available for this video: {e}")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [5]:
transcript_list

[{'text': 'the following is a conversation with', 'start': 0.08},
 {'text': 'demus hasabis', 'start': 1.76},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52},
 {'text': 'a company that has published and builds', 'start': 6.72},
 {'text': 'some of the most incredible artificial', 'start': 8.639},
 {'text': 'intelligence systems in the history of', 'start': 11.2},
 {'text': 'computing including alfred zero that', 'start': 13.2},
 {'text': 'learned', 'start': 16.0},
 {'text': 'all by itself to play the game of gold', 'start': 16.88},
 {'text': 'better than any human in the world and', 'start': 18.96},
 {'text': 'alpha fold two that solved protein', 'start': 21.439},
 {'text': 'folding', 'start': 24.56},
 {'text': 'both tasks considered nearly impossible', 'start': 25.68},
 {'text': 'for a very long time', 'start': 28.72},
 {'text': 'demus is widely considered to be one of', 'start': 31.119},
 {'text': 'the most brilliant and impactful humans', 'start': 33.12},
 {'text': 'in the 

## Step 1b - Indexing (Text Splitting)

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [7]:
len(chunks)

168

In [8]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2492.30it/s]


In [10]:
vector_store.index_to_docstore_id

{0: 'c684cb16-bd92-412f-8078-790fad2786f7',
 1: 'ac872b16-ac28-4881-ae36-09932f64b1dd',
 2: 'a0723f74-984d-49c5-851d-ff1dd49f7297',
 3: 'd6186381-9e16-4497-b641-b41b73ada567',
 4: 'f68d63f8-6406-44ef-9435-611c5b6f3ed8',
 5: '8cc3ce90-666f-466f-a1d5-a5d2573cbe11',
 6: 'b1ee1023-d733-41c3-903f-4664912403ac',
 7: 'c2251452-a14c-4765-b838-e1638109b8e7',
 8: 'db1fdb73-4866-48e3-9c15-7b3c20f20765',
 9: 'ac88ebdc-7541-4368-a993-ff0b0ae33624',
 10: '77ea8b5e-a802-4b9e-8e09-5d867fb37dae',
 11: '0a320fd5-ecf4-4c82-b81b-6cb7ef36236d',
 12: 'b4db1797-d355-44a8-9d73-b47e7453a966',
 13: 'c38eeaff-36b0-4ac0-9bc3-4fc808b78aa5',
 14: '2f89ecf1-7963-4734-b6fe-8b115b51759f',
 15: '721c14e4-0e87-4812-8bb1-6bffbaa303f6',
 16: '5b98b39f-1efa-4cba-87d4-bd7509b7bb9b',
 17: 'd6d97e53-3542-49a3-acba-0ebaae1a9bc2',
 18: '840e0f86-80a4-4cbe-a69f-aaa6a48fe6ee',
 19: 'ed06ea7a-f2dd-4a2b-b4a6-82f56e0164b3',
 20: '189c50ea-489b-4a6d-8e5a-c508ea97f475',
 21: '79ae97b7-0937-4027-bdbb-c6fe2701e849',
 22: '9aa0ad84-c3f7-

In [11]:
vector_store.get_by_ids(['2436bdb8-3f5f-49c6-8915-0c654c888700'])

[]

## Step 2 - Retrieval

In [12]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [13]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001CD2E05B4D0>, search_kwargs={'k': 4})

In [14]:
retriever.invoke('What is deepmind')

[Document(id='f20f3004-2b6e-43a6-a957-b219d096d055', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

## Step 3 - Augmentation

In [ ]:
import os
from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface.chat_models import ChatHuggingFace

os.environ["HUGGINGFACE_HUB_API_TOKEN"] = "hf_--------------------"

llm = ChatHuggingFace(
    llm=HuggingFaceEndpoint(
        repo_id="Qwen/Qwen2.5-72B-Instruct",
        task="text-generation",
        huggingfacehub_api_token=os.environ["HUGGINGFACE_HUB_API_TOKEN"],
    )
)


In [16]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [17]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [18]:
retrieved_docs

[Document(id='76807503-234f-45bd-8463-d0dd81364d51', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [19]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [20]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [21]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we 

## Step 4 - Generation

In [22]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of nuclear fusion is discussed in the provided transcript. The discussion revolves around the application of AI to address some of the challenges in nuclear fusion, particularly focusing on:

1. **Collaboration with EPFL**: The speaker mentions collaborating with EPFL (École Polytechnique Fédérale de Lausanne) in Switzerland, which has a test reactor. They were allowed to use this reactor for their experiments, which is described as an "amazing test reactor" where they conduct various experiments.

2. **Bottleneck Problems**: The team identifies bottleneck problems in fusion, such as physics, material science, and engineering challenges. They focus on problems that can be addressed using AI methods.

3. **Plasma Control**: One of the significant achievements discussed is the use of AI to control and hold plasma in specific shapes for extended periods. This was achieved using a controller that can maintain the plasma in different shapes, which is crucial for energy produc

## Building a Chain

In [23]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [24]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [25]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [26]:
parallel_chain.invoke('who is Demis')

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on

In [27]:
parser = StrOutputParser()

In [28]:
main_chain = parallel_chain | prompt | llm | parser

In [29]:
main_chain.invoke('Can you summarize the video')

"The provided text seems to be a partial and somewhat disjointed transcript from a conversation or interview. However, based on the content, here’s a summary:\n\nThe conversation appears to revolve around the nature of explanations, particularly in the realm of physics and AI. The speaker suggests that a deeper, perhaps simpler explanation of the physical world is needed, one that goes beyond the current standard model of physics. They propose that such an explanation might provide insights into fundamental questions like consciousness, life, and gravity.\n\nThe discussion also touches on the importance of clear and simple explanations, referencing Richard Feynman's idea that the ability to explain complex topics simply is a sign of true understanding. The speaker reflects on the relationship between humans and technology, noting that we are already symbiotic with our devices (e.g., phones).\n\nAdditionally, the conversation includes a historical anecdote about early computing, specifi